In [2]:
!pip install river==0.25.0 openpyxl -q

In [3]:
import pandas as pd
import numpy as np
import math
import numbers
import re
from collections import defaultdict, Counter
from river import (compose,preprocessing,linear_model,imblearn,metrics)

In [4]:
history = pd.read_excel("FundTransferData1K_Labeled.xlsx")
current = pd.read_excel("Current_1K_From_FundTransfer_History.xlsx")

In [5]:
print("Historical shape:", history.shape)
print("Current shape:", current.shape)

Historical shape: (1025, 16)
Current shape: (1000, 102)


In [6]:
print("\nHistorical labels:")
print(history["FRAUD_LABEL"].value_counts())


Historical labels:
FRAUD_LABEL
0    977
1     48
Name: count, dtype: int64


##Check users

In [7]:
print("Historical users:")
print(history["USER_ID"].value_counts())
print("\nCurrent users:")
print(current["USER_ID"].value_counts())

Historical users:
USER_ID
zubair             591
matiar31           213
towfiqur            99
01308242792         88
atiqur548833820     34
Name: count, dtype: int64

Current users:
USER_ID
atiqur548833820    200
towfiqur           200
01308242792        200
matiar31           200
zubair             200
Name: count, dtype: int64


##Create date parser

In [8]:
def parse_mixed_datetime(series):
    result = pd.Series(pd.NaT,index=series.index,dtype="datetime64[ns]")
    numeric = pd.to_numeric(series,errors="coerce")
    numeric_mask = numeric.notna()
    # Excel numeric dates
    result.loc[numeric_mask] = (pd.Timestamp("1899-12-30")+pd.to_timedelta(numeric.loc[numeric_mask],unit="D"))
    # Normal date strings
    text_mask = ~numeric_mask & series.notna()
    result.loc[text_mask] = pd.to_datetime(series.loc[text_mask],errors="coerce",dayfirst=True)
    return result

##Create cleaning functions

In [9]:
def clean_text(value):
    if pd.isna(value):
        return "__MISSING__"
    value = str(value).strip().lower()
    if value == "":
        return "__MISSING__"
    return value

In [10]:
def clean_id(value):
    if pd.isna(value):
        return "__MISSING__"
    # Integer
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    # Float like 1711159000.0
    if isinstance(value, (float, np.floating)):
        if np.isfinite(value) and value.is_integer():
            return str(int(value))
        return str(value).strip().lower()
    value = str(value).strip().lower()
    # Remove .0 from something like 123456.0
    if re.fullmatch(r"\d+\.0", value):
        value = value[:-2]
    return value if value else "__MISSING__"

##Convert historical data into a common format

In [11]:
history_stream = pd.DataFrame()
history_stream["ROW_ID"] = range(len(history))
history_stream["USER_ID"] = history["USER_ID"].apply(clean_id)
history_stream["DT"] = parse_mixed_datetime(history["TXN_TIME"])
history_stream["AMOUNT"] = pd.to_numeric(history["AMOUNT"],errors="coerce").fillna(0.0)
history_stream["RECEIVER_ID"] = (history["RECEIVER_ID"].apply(clean_id))
history_stream["RECEIVER_NM"] = (history["RECEIVER_NM"].apply(clean_text))
history_stream["TO_ACCOUNT_NO"] = (history["TO_ACCOUNT_NO"].apply(clean_id))
history_stream["PURPOSE"] = (history["PURPOSE"].apply(clean_text))
history_stream["TRANSFER_TYPE"] = (history["TRANSFER_TYPE"].apply(clean_text))
history_stream["FRAUD_LABEL"] = (pd.to_numeric(history["FRAUD_LABEL"],errors="coerce").fillna(0).astype(int))

In [12]:
print(history_stream.head())

   ROW_ID      USER_ID         DT    AMOUNT  RECEIVER_ID       RECEIVER_NM  \
0       0  01308242792 2024-06-30   84000.0   1711158900   srabon enterpri   
1       1  01308242792 2024-12-09   50000.0  __MISSING__  mr. habib  sabab   
2       2  01308242792 2024-12-22  240000.0  __MISSING__  mr. habib  sabab   
3       3  01308242792 2025-11-02  100000.0  __MISSING__  mr. habib  sabab   
4       4  01308242792 2025-04-21   13000.0  __MISSING__  mr. habib  sabab   

   TO_ACCOUNT_NO      PURPOSE    TRANSFER_TYPE  FRAUD_LABEL  
0  1729000037005  air tickets              eft            0  
1  1101007079707       urgent  ownbankotheracc            0  
2  1101007079707         gift  ownbankotheracc            1  
3  1101007079707     business  ownbankotheracc            1  
4  1101007079707         card  ownbankotheracc            0  


##Convert Current 1K into exactly the same structure

In [13]:
current_stream = pd.DataFrame()
current_stream["ROW_ID"] = range(len(current))
current_stream["USER_ID"] = (current["USER_ID"].apply(clean_id))
current_stream["DT"] = pd.to_datetime(current["EVENT_TIME"],errors="coerce",dayfirst=True)
current_stream["AMOUNT"] = pd.to_numeric(current["AMOUNT_LCY"],errors="coerce").fillna(0.0)
current_stream["RECEIVER_ID"] = (current["RECEIVER_ID"].apply(clean_id))
current_stream["RECEIVER_NM"] = (current["RECEIVER_NM"].apply(clean_text))
current_stream["TO_ACCOUNT_NO"] = (current["TO_ACCOUNT_NO"].apply(clean_id))
current_stream["PURPOSE"] = (current["PURPOSE"].apply(clean_text))
current_stream["TRANSFER_TYPE"] = (current["TRANSFER_TYPE"].apply(clean_text))

In [14]:
print(current_stream.head())

   ROW_ID          USER_ID                  DT  AMOUNT  RECEIVER_ID  \
0       0  atiqur548833820 2025-12-03 05:46:00   18990   1721548833   
1       1         towfiqur 2025-12-04 01:28:00    6000  __MISSING__   
2       2  atiqur548833820 2025-12-04 04:55:00   22000  __MISSING__   
3       3         towfiqur 2025-12-04 17:47:00     440  __MISSING__   
4       4  atiqur548833820 2025-12-04 20:39:00     300  __MISSING__   

               RECEIVER_NM  TO_ACCOUNT_NO     PURPOSE    TRANSFER_TYPE  
0           m a s builders  3001100001175   eft_trans              eft  
1  mr. md. a. rahman molla  1108000406710         sir  ownbankotheracc  
2            msmasbuilders  3001100001175  npsb_trans          npsba2a  
3      md. towfiqur rahman    01568783700          my            bkash  
4    atiqur rahaman naymee    01721548833    business            bkash  


##Sort historical transactions chronologically

In [15]:
# Sort historical transactions chronologically
history_stream = (
    history_stream
    .sort_values(["DT", "ROW_ID"])
    .reset_index(drop=True)
)

n = len(history_stream)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_stream = history_stream.iloc[:train_end].copy()
val_stream = history_stream.iloc[train_end:val_end].copy()
test_stream = history_stream.iloc[val_end:].copy()

print("Total:", len(history_stream))
print("Train:", len(train_stream))
print("Validation:", len(val_stream))
print("Test:", len(test_stream))

Total: 1025
Train: 717
Validation: 154
Test: 154


##Create user-history memory

In [16]:
def new_user_state():
    return {"count": 0,
            "amount_sum": 0.0,
            "amount_sq_sum": 0.0,
            "max_amount": 0.0,
            "last_time": None,
            "receivers": Counter(),
            "accounts": Counter(),
            "purposes": Counter(),
            "transfer_types": Counter()
            }
user_states = defaultdict(new_user_state)

##Create River fraud features

In [17]:
def build_features(row, states):
    user = row["USER_ID"]
    state = states[user]
    amount = float(row["AMOUNT"])
    dt = row["DT"]
    receiver_key = (
        row["RECEIVER_ID"]
        if row["RECEIVER_ID"] != "__MISSING__"
        else row["RECEIVER_NM"]
    )
    account = row["TO_ACCOUNT_NO"]
    purpose = row["PURPOSE"]
    transfer_type = row["TRANSFER_TYPE"]

    # Previous user statistics
    count = state["count"]
    if count > 0:
        avg_amount = (state["amount_sum"] / count)
        variance = (state["amount_sq_sum"] / count - avg_amount ** 2)
        variance = max(variance,0)
        std_amount = math.sqrt(variance)
    else:
        avg_amount = 0.0
        std_amount = 0.0

    # Amount relative to history
    if avg_amount > 0:
      amount_ratio = (amount / avg_amount)
    else:
      amount_ratio = 1.0

    # Time since last transaction

    if (state["last_time"] is not None and pd.notna(dt)):
        hours_since_last = ((dt - state["last_time"]).total_seconds()/ 3600)
        hours_since_last = max(hours_since_last,0.0)
    else:
        hours_since_last = 0.0

    # Time features

    if pd.notna(dt):
        hour = int(dt.hour)
        day_of_week = int(dt.dayofweek)
    else:
        hour = 0
        day_of_week = 0

    is_weekend = int(day_of_week in [5, 6])
    is_night = int(hour in [0, 1, 2, 3, 4, 5])

    # Previous behavior

    receiver_count = (state["receivers"][receiver_key])
    account_count = (state["accounts"][account])
    purpose_count = (state["purposes"][purpose])
    transfer_count = (state["transfer_types"][transfer_type])

    # Final River feature dict

    x = {

    "amount": amount,

    "hour_of_day": hour,

    "day_of_week": day_of_week,

    "is_weekend": is_weekend,

    "is_night": is_night,

    "user_txn_count_before": count,

    "user_avg_amount_before": avg_amount,

    "user_std_amount_before": std_amount,

    "user_max_amount_before": state["max_amount"],

    "amount_to_user_avg": amount_ratio,

    "hours_since_last_txn": hours_since_last,

    "receiver_count_before": receiver_count,

    "receiver_seen_before": int(
        receiver_count > 0
    ),

    "new_receiver": int(
        receiver_count == 0
    ),

    "account_count_before": account_count,

    "account_seen_before": int(
        account_count > 0
    ),

    "new_account": int(
        account_count == 0
    ),

    "purpose_count_before": purpose_count,

    "purpose_seen_before": int(
        purpose_count > 0
    ),

    # FIXED HERE
    "transfer_type_count_before": transfer_count,

    "transfer_type_seen_before": int(
        transfer_count > 0
    ),

    "purpose": purpose,

    "transfer_type": transfer_type
}

    return x

##Update user behavior after each transaction

In [18]:
def update_user_state(row, states):

    user = row["USER_ID"]

    state = states[user]

    amount = float(row["AMOUNT"])

    receiver_key = (row["RECEIVER_ID"]
        if row["RECEIVER_ID"] != "__MISSING__"
        else row["RECEIVER_NM"]
    )

    account = row["TO_ACCOUNT_NO"]
    purpose = row["PURPOSE"]
    transfer_type = row["TRANSFER_TYPE"]
    state["count"] += 1
    state["amount_sum"] += amount
    state["amount_sq_sum"] += (amount ** 2)
    state["max_amount"] = max(state["max_amount"],amount)
    if pd.notna(row["DT"]):
        state["last_time"] = (row["DT"])
    state["receivers"][receiver_key] += 1
    state["accounts"][account] += 1
    state["purposes"][purpose] += 1
    state["transfer_types"][transfer_type] += 1

##Build the River

In [19]:
numeric_features = (compose.SelectType(numbers.Number)|preprocessing.StandardScaler())
categorical_features = (compose.SelectType(str)|preprocessing.OneHotEncoder())

In [20]:
river_classifier = (numeric_features + categorical_features|linear_model.LogisticRegression())

In [21]:
model = imblearn.RandomOverSampler(classifier=river_classifier,desired_dist={False: 0.50,True: 0.50},seed=42)

##Create River evaluation metrics

In [22]:
accuracy = metrics.Accuracy()

precision = metrics.Precision(
    pos_val=True
)

recall = metrics.Recall(
    pos_val=True
)

f1 = metrics.F1(
    pos_val=True
)

roc_auc = metrics.ROCAUC(
    n_thresholds=50
)

confusion = metrics.ConfusionMatrix()


In [23]:
numeric_features = (
    compose.SelectType(numbers.Number)
    |
    preprocessing.StandardScaler()
)

categorical_features = (
    compose.SelectType(str)
    |
    preprocessing.OneHotEncoder()
)

river_classifier = (
    numeric_features
    +
    categorical_features
    |
    linear_model.LogisticRegression()
)

model = imblearn.RandomOverSampler(
    classifier=river_classifier,
    desired_dist={
        False: 0.50,
        True: 0.50
    },
    seed=42
)

##Train River using historical transactions

In [24]:
evaluated = 0

for index, row in history_stream.iterrows():

    x = build_features(
        row,
        user_states
    )

    y = bool(
        row["FRAUD_LABEL"]
    )

    probability = model.predict_proba_one(x)

    fraud_score = probability.get(
        True,
        0.0
    )

    y_pred = model.predict_one(x)

    if y_pred is not None:

        accuracy.update(y, y_pred)
        precision.update(y, y_pred)
        recall.update(y, y_pred)
        f1.update(y, y_pred)
        roc_auc.update(y, fraud_score)
        confusion.update(y, y_pred)

        evaluated += 1

    model.learn_one(
        x,
        y
    )

    update_user_state(
        row,
        user_states
    )

##River's historical streaming performance

In [25]:
print(
    "Evaluated transactions:",
    evaluated
)

print(
    "Accuracy:",
    accuracy.get()
)

print(
    "Precision:",
    precision.get()
)

print(
    "Recall:",
    recall.get()
)

print(
    "F1:",
    f1.get()
)

print(
    "ROC-AUC:",
    roc_auc.get()
)

print("\nConfusion Matrix:")

print(
    confusion
)

Evaluated transactions: 1025
Accuracy: 0.8848780487804878
Precision: 0.25
Recall: 0.7291666666666666
F1: 0.3723404255319149
ROC-AUC: 0.902667604912999

Confusion Matrix:
        False   True  
False     872    105  
 True      13     35  


In [26]:
import copy
import pickle
from pathlib import Path

river_api_bundle = {

    "model":
        copy.deepcopy(model),

    "user_states":
        copy.deepcopy(
            dict(user_states)
        ),

    "trained_through":
        pd.Timestamp(
            history_stream["DT"].max()
        ).isoformat(),

    "historical_count":
        len(history_stream),

    "model_type":
        "River online supervised fraud classifier",

    "risk_high":
        0.75,

    "risk_medium":
        0.50
}

MODEL_PATH = (
    Path.cwd()
    / "river_api_bundle.pkl"
)

with open(MODEL_PATH, "wb") as file:

    pickle.dump(
        river_api_bundle,
        file
    )

print("River API bundle created!")

print(
    "Saved location:",
    MODEL_PATH.resolve()
)

print(
    "File exists:",
    MODEL_PATH.exists()
)

River API bundle created!
Saved location: C:\Users\ASUS\OneDrive\Desktop\river\River_API_VSCode\river_api_bundle.pkl
File exists: True


##Prepare Current 1K as a chronological stream

In [27]:
current_stream = (current_stream.sort_values(["DT", "ROW_ID"]).reset_index(drop=True))

##Predict Current 1K using River

In [28]:
river_predictions = []

In [29]:
for index, row in current_stream.iterrows():

    # Build features from previous behavior
    x = build_features(
        row,
        user_states
    )

    # River probability
    probability = (
        model.predict_proba_one(x)
    )

    fraud_score = probability.get(
        True,
        0.0
    )

    normal_score = probability.get(
        False,
        0.0
    )

    # River predicted class
    prediction = model.predict_one(x)

    if prediction is None:

        prediction = False

    predicted_fraud = int(
        prediction
    )

    river_predictions.append({

        "ROW_ID":
            row["ROW_ID"],

        "RIVER_NORMAL_SCORE":
            normal_score,

        "RIVER_FRAUD_SCORE":
            fraud_score,

        "RIVER_PREDICTED_FRAUD":
            predicted_fraud
    })

    # ----------------------------------
    # Update transaction behavior only
    # ----------------------------------
    #
    # We KNOW the transaction happened,
    # even though we do NOT know whether
    # it was truly fraud.
    #
    # So behavioral history can update.
    # The classifier itself DOES NOT learn.
    # ----------------------------------

    update_user_state(
        row,
        user_states
    )

##Convert predictions into DataFrame

In [30]:
river_predictions = pd.DataFrame(
    river_predictions
)

In [31]:
print(
    river_predictions.head()
)

   ROW_ID  RIVER_NORMAL_SCORE  RIVER_FRAUD_SCORE  RIVER_PREDICTED_FRAUD
0       8            0.966792           0.033208                      0
1       0            0.857369           0.142631                      0
2       1            0.823412           0.176588                      0
3       2            0.949536           0.050464                      0
4       3            0.988062           0.011938                      0


##Restore the original Excel row order

In [32]:
river_predictions = (
    river_predictions
    .sort_values("ROW_ID")
    .reset_index(drop=True)
)

##Attach predictions to Current 1K

In [33]:
final_result = current.copy()

final_result[
    "RIVER_NORMAL_SCORE"
] = river_predictions[
    "RIVER_NORMAL_SCORE"
]

final_result[
    "RIVER_FRAUD_SCORE"
] = river_predictions[
    "RIVER_FRAUD_SCORE"
]

final_result[
    "RIVER_PREDICTED_FRAUD"
] = river_predictions[
    "RIVER_PREDICTED_FRAUD"
]

##Add an easy risk catagory

In [34]:
def river_risk_level(score):

    if score >= 0.75:
        return "HIGH"

    elif score >= 0.50:
        return "MEDIUM"

    else:
        return "LOW"

In [35]:
final_result[
    "RIVER_RISK_LEVEL"
] = final_result[
    "RIVER_FRAUD_SCORE"
].apply(
    river_risk_level
)

##Count predicted frauds

In [36]:
print(
    final_result[
        "RIVER_PREDICTED_FRAUD"
    ].value_counts()
)

RIVER_PREDICTED_FRAUD
0    926
1     74
Name: count, dtype: int64


##Show only suspected fraud transactions

In [37]:
fraud_transactions = (
    final_result[
        final_result[
            "RIVER_PREDICTED_FRAUD"
        ] == 1
    ]
    .sort_values(
        "RIVER_FRAUD_SCORE",
        ascending=False
    )
)

In [38]:
print(
    fraud_transactions[
        [
            "EVENT_ID",
            "USER_ID",
            "EVENT_TIME",
            "AMOUNT_LCY",
            "RECEIVER_NM",
            "TO_ACCOUNT_NO",
            "PURPOSE",
            "RIVER_FRAUD_SCORE",
            "RIVER_PREDICTED_FRAUD",
            "RIVER_RISK_LEVEL"
        ]
    ].head(30)
)

     EVENT_ID      USER_ID           EVENT_TIME  AMOUNT_LCY  \
570   1975607  01308242792  19-04-2026 14:35:00      601930   
738   1975654  01308242792  29-05-2026 10:27:00      640040   
999   1976461       zubair  30-05-2027 18:32:00      100000   
957   1976419       zubair  24-04-2027 13:11:00      376450   
846   1976308       zubair  17-01-2027 03:42:00      100000   
932   1976394       zubair  02-04-2027 14:39:00      100000   
315   1975541  01308242792  19-02-2026 01:34:00      444400   
837   1976299       zubair  08-01-2027 07:57:00      100000   
919   1976381       zubair  21-03-2027 22:43:00          10   
444   1975573  01308242792  21-03-2026 15:30:00      300930   
845   1976307       zubair  16-01-2027 07:59:00      233870   
388   1975558  01308242792  07-03-2026 23:48:00      265000   
372   1975555  01308242792  04-03-2026 13:29:00      358000   
747   1975656  01308242792  31-05-2026 14:06:00      293500   
507   1975589  01308242792  05-04-2026 14:37:00       9

##Top 20 highest-risk transactions

In [39]:
top_20 = (
    final_result
    .sort_values(
        "RIVER_FRAUD_SCORE",
        ascending=False
    )
    .head(20)
)

In [40]:
print(
    top_20[
        [
            "EVENT_ID",
            "USER_ID",
            "EVENT_TIME",
            "AMOUNT_LCY",
            "RECEIVER_NM",
            "PURPOSE",
            "RIVER_FRAUD_SCORE",
            "RIVER_PREDICTED_FRAUD"
        ]
    ]
)

     EVENT_ID      USER_ID           EVENT_TIME  AMOUNT_LCY  \
570   1975607  01308242792  19-04-2026 14:35:00      601930   
738   1975654  01308242792  29-05-2026 10:27:00      640040   
999   1976461       zubair  30-05-2027 18:32:00      100000   
957   1976419       zubair  24-04-2027 13:11:00      376450   
846   1976308       zubair  17-01-2027 03:42:00      100000   
932   1976394       zubair  02-04-2027 14:39:00      100000   
315   1975541  01308242792  19-02-2026 01:34:00      444400   
837   1976299       zubair  08-01-2027 07:57:00      100000   
919   1976381       zubair  21-03-2027 22:43:00          10   
444   1975573  01308242792  21-03-2026 15:30:00      300930   
845   1976307       zubair  16-01-2027 07:59:00      233870   
388   1975558  01308242792  07-03-2026 23:48:00      265000   
372   1975555  01308242792  04-03-2026 13:29:00      358000   
747   1975656  01308242792  31-05-2026 14:06:00      293500   
507   1975589  01308242792  05-04-2026 14:37:00       9

##User-wise fraud summary

In [41]:
user_summary = (
    final_result
    .groupby("USER_ID")
    .agg(

        TOTAL_TRANSACTIONS=(
            "EVENT_ID",
            "count"
        ),

        PREDICTED_FRAUDS=(
            "RIVER_PREDICTED_FRAUD",
            "sum"
        ),

        AVG_FRAUD_SCORE=(
            "RIVER_FRAUD_SCORE",
            "mean"
        ),

        MAX_FRAUD_SCORE=(
            "RIVER_FRAUD_SCORE",
            "max"
        )

    )
    .reset_index()
)

In [42]:
print(user_summary)

           USER_ID  TOTAL_TRANSACTIONS  PREDICTED_FRAUDS  AVG_FRAUD_SCORE  \
0      01308242792                 200                21         0.228836   
1  atiqur548833820                 200                 5         0.082019   
2         matiar31                 200                 0         0.026626   
3         towfiqur                 200                 0         0.029581   
4           zubair                 200                48         0.230747   

   MAX_FRAUD_SCORE  
0         0.998512  
1         0.745290  
2         0.488834  
3         0.296162  
4         0.991366  
